# Stage 1 grouped reporting

This notebook shows how to use `reporting.py` with the existing solver results. It keeps the solver unchanged and focuses only on output formatting.

The main ideas are:

- group element tables by `region` and `subregion`;
- aggregate split FE sub-elements through `physical_member_id`;
- export a cleaner XLSX workbook with one sheet per major region.


In [ ]:
from pathlib import Path
import sys
import pandas as pd

CWD = Path.cwd().resolve()
if (CWD / 'solver.py').exists():
    PROJECT_ROOT = CWD
elif (CWD.parent / 'solver.py').exists():
    PROJECT_ROOT = CWD.parent
else:
    raise FileNotFoundError('Could not find solver.py in the current directory or one level above.')

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_DIR = PROJECT_ROOT / 'data'
RESULTS_DIR = PROJECT_ROOT / 'results'
RESULTS_DIR.mkdir(exist_ok=True)

from solver import read_model, SolveOptions, solve_model, summarize_result
from reporting import (
    enrich_results_for_reporting,
    region_summary_table,
    physical_member_summary_table,
    all_elements_grouped_table,
    compression_member_table,
    frame_bending_table,
    export_grouped_results_to_excel,
    DEFAULT_GROUPED_DISPLAY_COLS,
)


## Analysis options and load case

Use the same load-case format as the main workflow notebook.


In [ ]:
SECTION_MODE = 'principal'
CROSSING_DIAGONAL_MODE = 'continuous'
TOP_N_CRITICAL = 20

LOAD_CASE_BY_LABEL = [
    {'node_label': 'high_xneg', 'Fx': 17000.0, 'Fy': 2000.0, 'Fz': -23400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
    {'node_label': 'high_xpos', 'Fx': 17000.0, 'Fy': 2000.0, 'Fz': -23400.0, 'Mx': 0.0, 'My': 0.0, 'Mz': 0.0},
]


In [ ]:
models = {
    'TE5': read_model('TE5', DATA_DIR / 'TE5_nodes_aligned.csv', DATA_DIR / 'TE5_elements_with_sections.csv'),
    'Z5': read_model('Z5', DATA_DIR / 'Z5_nodes_aligned.csv', DATA_DIR / 'Z5_elements_with_sections.csv'),
}

options = SolveOptions(
    crossing_diagonal_mode=CROSSING_DIAGONAL_MODE,
    section_mode=SECTION_MODE,
)

results = {
    name: solve_model(model, LOAD_CASE_BY_LABEL, options)
    for name, model in models.items()
}
results = enrich_results_for_reporting(results)

pd.DataFrame([summarize_result(res) for res in results.values()])


## Grouped output tables


In [ ]:
print('Region summary')
display(region_summary_table(results))

print('Physical members: continuous-for-reporting aggregation')
display(physical_member_summary_table(results).head(40))

print('All elements grouped')
all_grouped = all_elements_grouped_table(results)
display(all_grouped[[c for c in DEFAULT_GROUPED_DISPLAY_COLS if c in all_grouped.columns]])


In [ ]:
print('Most compressive elements')
comp = compression_member_table(results)
display(comp[[c for c in DEFAULT_GROUPED_DISPLAY_COLS if c in comp.columns]].head(25))

print('Frame elements sorted by bending resultant')
bend = frame_bending_table(results)
display(bend[[c for c in DEFAULT_GROUPED_DISPLAY_COLS if c in bend.columns]].head(25))


## Export grouped workbook


In [ ]:
out_xlsx = RESULTS_DIR / f'stage1_grouped_report_{SECTION_MODE}_{CROSSING_DIAGONAL_MODE}.xlsx'
export_grouped_results_to_excel(results, out_xlsx, top_n=TOP_N_CRITICAL)
print('Wrote:', out_xlsx)
